The synthetic Big 5 data was generated based on this paper:

Weisberg, Y. J., DeYoung, C. G., & Hirsh, J. B. (2011). Gender differences in personality across the ten aspects of the Big Five. Frontiers in psychology, 2, 11757.

In [1]:
import numpy as np
import pandas as pd
np.random.seed(1)

facets = [
    "Enthusiasm", "Assertiveness",
    "Compassion", "Politeness",
    "Industriousness", "Orderliness",
    "Volatility", "Withdrawal",
    "Intellect", "Openness"
]

corr = np.array([
    # Enth. Asse.  Comp.  Poli.  Ind.   Order.  Volat. With.  Inte.  Open.
    [1.00,  0.45,  0.46,  0.10,  0.21,  0.00,  -0.18, -0.35,  0.17,  0.15],  # Enth.
    [0.51,  1.00,  0.22,  -0.20, 0.34,  0.06,  -0.09, -0.40,  0.47,  0.20],  # Asse.
    [0.45,  0.25,  1.00,  0.38,  0.14,  0.03,  -0.17, -0.12,  0.24,  0.43],  # Comp.
    [0.12,  -0.17, 0.47,  1.00,  0.29,  0.23,  -0.35, -0.20,  -0.01, 0.07],  # Poli
    [0.22,  0.36,  0.21,  0.28,  1.00,  0.41,  -0.24, -0.51,  0.33,  -0.02], # Ind.
    [0.04,  0.08,  0.14,  0.21,  0.44,  1.00,  0.03,  -0.03, -0.02,  -0.07], # Order.
    [-0.15, -0.07, -0.15, -0.40, -0.36, -0.02, 1.00,  0.61,  -0.23,  0.00],  # Volat.
    [-0.37, -0.42, -0.14, -0.17, -0.54, -0.03, 0.62,  1.00,  -0.35,  0.03],  # With.
    [0.22,  0.48,  0.31,  0.10,  0.42,  0.04,  -0.21, -0.35, 1.00,   0.38],  # Inte.
    [0.13,  0.17,  0.39,  0.17,  0.01,  -0.03, 0.01,  0.04,  0.42,   1.00]   # Open.
])

# This splits the correlation matrix into the upper an lower triangle (above for male, below for women)
corr_male = corr.copy()
corr_female = corr.copy()
for i in range(len(facets)):
    for j in range(len(facets)):
        if i > j:
            corr_male[i, j] = corr[j, i]     # mirror upper triangle
        if i < j:
            corr_female[i, j] = corr[j, i]   # mirror lower triangle

n = 1000
p_male = 0.5
n_m = int(n * p_male)
n_f = n - n_m

Z_m = np.random.multivariate_normal(
    mean=np.zeros(len(facets)),
    cov=corr_male,
    size=n_m
)

Z_f = np.random.multivariate_normal(
    mean=np.zeros(len(facets)),
    cov=corr_female,
    size=n_f
)

df_m = pd.DataFrame(Z_m, columns=facets)
df_m["gender"] = "male"

df_f = pd.DataFrame(Z_f, columns=facets)
df_f["gender"] = "female"

df = pd.concat([df_m, df_f], ignore_index=True)
params = {
    "Enthusiasm":     {"male": (3.40, 0.66), "female": (3.56, 0.68)},
    "Assertiveness":  {"male": (3.34, 0.64), "female": (3.28, 0.68)},
    "Compassion":     {"male": (3.78, 0.60), "female": (4.04, 0.56)},
    "Politeness":     {"male": (3.52, 0.61), "female": (3.74, 0.61)},
    "Industriousness":{"male": (3.25, 0.68), "female": (3.21, 0.73)},
    "Orderliness":    {"male": (3.38, 0.61), "female": (3.49, 0.63)},
    "Volatility":     {"male": (2.63, 0.75), "female": (2.86, 0.77)},
    "Withdrawal":     {"male": (2.73, 0.69), "female": (3.02, 0.71)},
    "Intellect":      {"male": (3.62, 0.61), "female": (3.48, 0.63)},
    "Openness":       {"male": (3.57, 0.61), "female": (3.74, 0.60)}
}

for facet, gender_params in params.items():
    for gender, (mu, sd) in gender_params.items():
        idx = df["gender"] == gender
        z = (df.loc[idx, facet] - df.loc[idx, facet].mean()) / df.loc[idx, facet].std()
        df.loc[idx, facet] = mu + sd * z

# clip values between 1-5 (This is a common range for Big 5 Tests)        
for facet in facets:
    df[facet] = df[facet].clip(lower=1, upper=5)

df["Extraversion"] = df[["Enthusiasm", "Assertiveness"]].mean(axis=1)
df["Agreeableness"] = df[["Compassion", "Politeness"]].mean(axis=1)
df["Conscientiousness"] = df[["Industriousness", "Orderliness"]].mean(axis=1)
df["Neuroticism"] = df[["Volatility", "Withdrawal"]].mean(axis=1)
df["Openness_Intellect"] = df[["Intellect", "Openness"]].mean(axis=1)

In [ ]:
# df.to_csv("Synthetic_Big5_data.csv", index=False)

# Tests:

In [22]:
df.describe()

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,3.477344,3.308649,3.900811,3.627546,3.227623,3.433953,2.749624,2.875602,3.548425,3.653145,3.392996,3.764178,3.330788,2.812613,3.600785
std,0.667927,0.656367,0.573587,0.613298,0.698241,0.619350,0.755868,0.710062,0.619540,0.606253,0.570075,0.508226,0.548730,0.655403,0.508833
min,1.175951,1.371449,1.799046,1.812669,1.236628,1.584233,1.000000,1.000000,1.763540,1.786451,1.729169,2.168681,1.880809,1.000000,1.952700
25%,3.005816,2.855570,3.533009,3.215809,2.758117,2.998351,2.213935,2.409944,3.131314,3.273837,3.001044,3.432072,2.929372,2.347985,3.245803
50%,3.483260,3.321810,3.918751,3.638544,3.210418,3.423913,2.755692,2.875472,3.541471,3.665453,3.398068,3.782634,3.338290,2.804512,3.620321
75%,3.939801,3.758329,4.305758,4.070020,3.714479,3.837509,3.282291,3.330236,3.980844,4.069027,3.811018,4.120306,3.707308,3.258898,3.933728
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.868132,4.915163,5.000000


In [23]:
df[df["gender"] == "male"].describe().loc[['mean', 'std']]

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
mean,3.398315,3.34,3.771599,3.519260,3.249230,3.378813,2.634079,2.730655,3.617637,3.568674,3.369157,3.645429,3.314022,2.682367,3.593156
std,0.655490,0.64,0.579474,0.607983,0.677845,0.606526,0.739797,0.688230,0.603774,0.606556,0.547894,0.500455,0.531728,0.630024,0.497649


In [24]:
df[df["gender"] == "female"].describe().loc[['mean', 'std']]

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
mean,3.556373,3.277298,4.030023,3.735832,3.206016,3.489093,2.865169,3.020550,3.479213,3.737615,3.416835,3.882927,3.347554,2.942859,3.608414
std,0.671538,0.671512,0.537973,0.599900,0.718087,0.627686,0.754826,0.702583,0.627921,0.594654,0.591013,0.488253,0.565257,0.655011,0.520164


In [25]:
df[df["gender"] == "male"].drop(columns="gender").corr()

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
Enthusiasm,1.000000,0.430835,0.428380,0.076003,0.128093,-0.112958,-0.182876,-0.391380,0.086347,0.157327,0.849822,0.294176,0.017222,-0.321139,0.148259
Assertiveness,0.430835,1.000000,0.211243,-0.166582,0.362374,0.029084,-0.081553,-0.440476,0.469134,0.184056,0.841777,0.021112,0.247565,-0.288466,0.396757
Compassion,0.428380,0.211243,1.000000,0.420631,0.059565,-0.053154,-0.106835,-0.083072,0.182032,0.472879,0.379631,0.834452,0.007651,-0.108098,0.398609
Politeness,0.076003,-0.166582,0.420631,1.000000,0.176434,0.117680,-0.339297,-0.186291,-0.036227,0.074644,-0.051829,0.850954,0.179576,-0.300958,0.023514
Industriousness,0.128093,0.362374,0.059565,0.176434,1.000000,0.369212,-0.231228,-0.528211,0.358962,-0.034822,0.288270,0.141657,0.847973,-0.424263,0.196534
Orderliness,-0.112958,0.029084,-0.053154,0.117680,0.369212,1.000000,0.110455,0.043420,0.015178,-0.074463,-0.050584,0.040709,0.805671,0.088566,-0.036172
Volatility,-0.182876,-0.081553,-0.106835,-0.339297,-0.231228,0.110455,1.000000,0.556574,-0.172378,0.104932,-0.157026,-0.267951,-0.084388,0.891116,-0.040621
Withdrawal,-0.391380,-0.440476,-0.083072,-0.186291,-0.528211,0.043420,0.556574,1.000000,-0.336265,0.060277,-0.491382,-0.161254,-0.311917,0.872968,-0.167253
Intellect,0.086347,0.469134,0.182032,-0.036227,0.358962,0.015178,-0.172378,-0.336265,1.000000,0.352466,0.325652,0.083382,0.237458,-0.284872,0.821428
Openness,0.157327,0.184056,0.472879,0.074644,-0.034822,-0.074463,0.104932,0.060277,0.352466,1.000000,0.201610,0.319113,-0.064664,0.094530,0.823238


In [26]:
df[df["gender"] == "female"].drop(columns="gender").corr()

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
Enthusiasm,1.000000,0.549170,0.392788,0.077164,0.267295,0.093832,-0.142317,-0.382429,0.249831,0.120797,0.880110,0.263798,0.221880,-0.287103,0.219841
Assertiveness,0.549170,1.000000,0.198001,-0.142995,0.436378,0.180105,-0.124269,-0.481831,0.506619,0.175184,0.880100,0.021236,0.377180,-0.330015,0.405921
Compassion,0.392788,0.198001,1.000000,0.471397,0.188311,0.154877,-0.140000,-0.073230,0.245973,0.349298,0.335638,0.840512,0.205604,-0.119941,0.348124
Politeness,0.077164,-0.142995,0.471397,1.000000,0.260001,0.181279,-0.388548,-0.172024,0.096352,0.175151,-0.037397,0.874034,0.265800,-0.316137,0.158273
Industriousness,0.267295,0.436378,0.188311,0.260001,1.000000,0.408690,-0.390237,-0.560511,0.470573,0.021103,0.399764,0.263471,0.862101,-0.525462,0.296091
Orderliness,0.093832,0.180105,0.154877,0.181279,0.408690,1.000000,-0.011915,-0.051515,0.079534,-0.022181,0.155626,0.196690,0.814817,-0.034494,0.035327
Volatility,-0.142317,-0.124269,-0.140000,-0.388548,-0.390237,-0.011915,1.000000,0.615443,-0.269771,-0.022955,-0.151451,-0.315826,-0.254489,0.906264,-0.175949
Withdrawal,-0.382429,-0.481831,-0.073230,-0.172024,-0.560511,-0.051515,0.615443,1.000000,-0.401413,0.013436,-0.490997,-0.146023,-0.384632,0.890928,-0.234605
Intellect,0.249831,0.506619,0.245973,0.096352,0.470573,0.079534,-0.269771,-0.401413,1.000000,0.447758,0.429747,0.194703,0.343061,-0.370723,0.859520
Openness,0.120797,0.175184,0.349298,0.175151,0.021103,-0.022181,-0.022955,0.013436,0.447758,1.000000,0.168151,0.300035,0.001089,-0.006020,0.841861
